##### Camerano Alessio - Matricola 1044444
##### Corso di Elaborazione di Immagini e Visione Artificiale
---
# Riconoscimento di Sagome Arbitrarie tramite R-Table
---

### Cella 1: Inizializzazione dell'Ambiente e Funzioni di Servizio

In [80]:
import time
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

### Cella 2: Ingestione Dati e Pre-elaborazione
Questo passaggio produce la silhouette binaria dell'oggetto tramite la sogliatura di Otsu invertita, isolando efficacemente il foreground dal fondo della scena. La maschera binaria così ottenuta costituisce il prerequisito fondamentale per la successiva estrazione dei bordi significativi, che verranno elaborati mediante operatori differenziali.

In [81]:
path_scene = Path('train')
file_scene = list(path_scene.glob('*.[gG][iI][fF]'))
print(f"Trovate {len(file_scene)} immagini .gif nella cartella '{path_scene}'\n")

immagini_originali = {}
immagini_binarizzate = {}

for filepath in file_scene:
    nome_file = filepath.name 
    try:
        with Image.open(filepath) as img_gif:
            img_gray = np.array(img_gif.convert('L'))
    except Exception as e:
        print(f"Errore nel caricamento di {nome_file}: {e}")
        continue
        
    _, img_thresh = cv2.threshold(img_gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    
    immagini_originali[nome_file] = img_gray
    immagini_binarizzate[nome_file] = img_thresh
    print(f"Caricata e binarizzata con successo: {nome_file}")


Trovate 180 immagini .gif nella cartella 'train'

Caricata e binarizzata con successo: Bone-3.gif
Caricata e binarizzata con successo: apple-1.gif
Caricata e binarizzata con successo: apple-2.gif
Caricata e binarizzata con successo: apple-3.gif
Caricata e binarizzata con successo: apple-4.gif
Caricata e binarizzata con successo: apple-5.gif
Caricata e binarizzata con successo: apple-6.gif
Caricata e binarizzata con successo: apple-7.gif
Caricata e binarizzata con successo: apple-8.gif
Caricata e binarizzata con successo: apple-9.gif
Caricata e binarizzata con successo: apple-10.gif
Caricata e binarizzata con successo: apple-11.gif
Caricata e binarizzata con successo: apple-12.gif
Caricata e binarizzata con successo: apple-13.gif
Caricata e binarizzata con successo: apple-14.gif
Caricata e binarizzata con successo: apple-15.gif
Caricata e binarizzata con successo: apple-16.gif
Caricata e binarizzata con successo: apple-17.gif
Caricata e binarizzata con successo: apple-18.gif
Caricata e 

### Cella 3: Estrazione Feature e Creazione R-Table
In questa cella viene costruita la R-Table collezionando i vettori geometrici che collegano i pixel del contorno al baricentro (reference point) di ciascun template. Al fine di attenuare l'aliasing sui bordi, l'operatore di Sobel viene applicato sull'immagine binaria sfocata; gli orientamenti del gradiente calcolati vengono successivamente quantizzati a gradi interi per mappare i vettori all'interno di un unico flat array bidimensionale. Questa struttura lineare cumulativa consente di accorpare i contributi di più modelli in un blocco di memoria contiguo, ottimizzato per operazioni di broadcasting.

In [82]:
path_template = Path('template')
files_template = list(path_template.glob('*.[gG][iI][fF]'))

if len(files_template) == 0:
    raise ValueError("La cartella 'template' è vuota! Inserisci i file .gif dei modelli.")

vettori_accumulati = []
print("--- AVVIO ADDESTRAMENTO COMPOSITO SELETTIVO ---")

BARICENTRO_GLOBALE_X = 0
BARICENTRO_GLOBALE_Y = 0
mappa_conteggi_template = {}

for filepath_template in files_template:
    nome_template_file = filepath_template.name
    
    try:
        with Image.open(filepath_template) as img_gif_temp:
            img_template_gray = np.array(img_gif_temp.convert('L'))
    except Exception as e:
        print(f"Errore nel caricamento del template {nome_template_file}: {e}")
        continue

    _, img_template_bin = cv2.threshold(img_template_gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    momenti = cv2.moments(img_template_bin)
    if momenti["m00"] == 0:
        continue
    cx_ref_locale = int(momenti["m10"] / momenti["m00"])
    cy_ref_locale = int(momenti["m01"] / momenti["m00"])

    r_spostamento_c = BARICENTRO_GLOBALE_X - cx_ref_locale
    r_spostamento_r = BARICENTRO_GLOBALE_Y - cy_ref_locale

    img_smooth = cv2.GaussianBlur(img_template_bin, (3, 3), 0)
    gx = cv2.Sobel(img_smooth, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(img_smooth, cv2.CV_64F, 0, 1, ksize=3)

    magnitudo = np.sqrt(gx**2 + gy**2)
    orientamento_deg = np.degrees(np.arctan2(gy, gx)) % 360

    max_mag = np.max(magnitudo) if np.max(magnitudo) > 0 else 1.0
    mappa_curvatura_stabile = magnitudo / max_mag

    pixel_bordo = magnitudo > 50
    righe_bordo, colonne_bordo = np.where(pixel_bordo)

    contatore_vettori_singolo = 0
    for r, c in zip(righe_bordo, colonne_bordo):
        phi = int(round(orientamento_deg[r, c])) % 360
        k_locale = mappa_curvatura_stabile[r, c]
        
        dr_locale = cy_ref_locale - r
        dc_locale = cx_ref_locale - c
        
        dr_globale = dr_locale + r_spostamento_r
        dc_globale = dc_locale + r_spostamento_c
        
        vettori_accumulati.append([phi, dr_globale, dc_globale, k_locale, len(mappa_conteggi_template)])
        contatore_vettori_singolo += 1
        
    mappa_conteggi_template[len(mappa_conteggi_template)] = contatore_vettori_singolo
    print(f"Modello Sottoparte ID {len(mappa_conteggi_template)-1} ('{nome_template_file}'): {contatore_vettori_singolo} vettori.")

R_table_np = np.array(vettori_accumulati, dtype=np.float32)
print(f"\nTraining completato. Vettori totali nella Super R-table: {R_table_np.shape}")


--- AVVIO ADDESTRAMENTO COMPOSITO SELETTIVO ---
Modello Sottoparte ID 0 ('Bone-1.gif'): 5654 vettori.
Modello Sottoparte ID 1 ('Bone-13.gif'): 4612 vettori.
Modello Sottoparte ID 2 ('Bone-15.gif'): 5219 vettori.
Modello Sottoparte ID 3 ('Bone-17.gif'): 6021 vettori.

Training completato. Vettori totali nella Super R-table: (21506, 5)


### Cella 4: Fase di Rilevamento e Spazio di Hough
Il processo di localizzazione proietta i vettori della R-Table all'interno di uno spazio di voto bidimensionale associato alla scena. L'implementazione integra un criterio previsto dall'estensione metodologica di Ballard (1981), ovvero la pesatura dei voti in funzione della magnitudo del gradiente rilevata sui contorni della scena, favorendo così le transizioni nette. Lo smoothing Gaussiano applicato alla matrice finale agisce come euristica implementativa volta ad ammorbidire l'accumulatore, riducendo la frammentazione dei voti intorno al picco principale. La metrica di confidenza adotta un denominatore normalizzato sull'estensione teorica massima della R-Table filtrata a 255, associata a una soglia empirica del 5% per massimizzare la sensibilità del sistema.

In [83]:
immagini_con_oggetto = []
totale_immagini = len(immagini_binarizzate)

# Soglia rigorosa basata sull'energia accumulata canonica
soglia_confidenza = 0.24  

phi_template = R_table_np[:, 0].astype(np.int32)
dr_template = R_table_np[:, 1]
dc_template = R_table_np[:, 2]
k_template = R_table_np[:, 3]
w_template = R_table_np[:, 4]

# Spazio dei parametri a quattro dimensioni: A(y, s, theta) (Sezione 4.5)
scale_test = [0.8, 1.0, 1.2]  
rotazioni_test = list(range(0, 360, 15)) 
stati_riflessione = [0, 1]              # Configurato: 0 = Normale, 1 = Specchiato
stati_inversione_colore = [False]         # Figura-sfondo standard (imposta [False, True] se necessario)

num_scale = len(scale_test)
num_rotazioni = len(rotazioni_test)

t_inizio_totale = time.time()
print("--- AVVIO ACCUMULATORE FORMALE BALLARD ---")

for nome_scena, img_scena_bin in immagini_binarizzate.items():
    t_scena = time.time()
    
    img_scena_smooth = cv2.GaussianBlur(img_scena_bin, (3, 3), 0)
    gx_scena = cv2.Sobel(img_scena_smooth, cv2.CV_64F, 1, 0, ksize=3)
    gy_scena = cv2.Sobel(img_scena_smooth, cv2.CV_64F, 0, 1, ksize=3)
    magnitudo_scena = np.sqrt(gx_scena**2 + gy_scena**2)
    orientamento_deg_scena = np.degrees(np.arctan2(gy_scena, gx_scena)) % 360

    # Soglia del modulo del gradiente g(x) per l'estrazione degli edge pixel (Sezione 2.1)
    pixel_bordo_scena = magnitudo_scena > 160 
    righe_scena, colonne_scena = np.where(pixel_bordo_scena)
    altezza, larghezza = img_scena_bin.shape

    if len(righe_scena) == 0:
        continue

    orientamenti_bordo = orientamento_deg_scena[pixel_bordo_scena].astype(np.int32)
    magnitudo_bordo = magnitudo_scena[pixel_bordo_scena]

    mappa_pixel_scena = {}
    for phi in range(360):
        scelta = (orientamenti_bordo == phi)
        if np.any(scelta):
            mappa_pixel_scena[phi] = (righe_scena[scelta], colonne_scena[scelta], magnitudo_bordo[scelta])

    accumulatore_4d = np.zeros((altezza, larghezza, num_scale, num_rotazioni), dtype=np.float32)
    meta_riflessione = {}

    for riflettere in stati_riflessione:
        for invertire_colore in stati_inversione_colore:
            
            # Trasformazioni canoniche applicate alla R-table
            phi_lavorazione = phi_template.copy()
            dc_lavorazione = dc_template.copy()
            dr_lavorazione = dr_template.copy()
            
            if riflettere == 1:
                dc_lavorazione = -dc_lavorazione
                phi_lavorazione = (180 - phi_lavorazione) % 360
                
            if invertire_colore:
                # Trasformazione Figura-Sfondo di Ballard: (phi + pi) mod 2pi
                phi_lavorazione = (phi_lavorazione + 180) % 360

            mappa_vettori_template = {}
            for phi in range(360):
                scelta_v = (phi_lavorazione == phi)
                if np.any(scelta_v):
                    mappa_vettori_template[phi] = (dr_lavorazione[scelta_v], dc_lavorazione[scelta_v], k_template[scelta_v], w_template[scelta_v])

            angoli_comuni = list(set(mappa_pixel_scena.keys()).intersection(set(range(360))))

            for idx_s, s in enumerate(scale_test):
                for idx_theta, theta in enumerate(rotazioni_test):
                    
                    theta_rad = np.radians(theta)
                    cos_t = np.cos(theta_rad)
                    sin_t = np.sin(theta_rad)
                    
                    acc_temporaneo_voto = np.zeros((altezza, larghezza), dtype=np.float32)

                    for phi_target in angoli_comuni:
                        r_scena_sub, c_scena_sub, mag_scena_sub = mappa_pixel_scena[phi_target]

                        # Eq. 19: Spostamento dell'indice (phi - theta) mod 2pi
                        phi_modificato_template = (phi_target - theta) % 360
                        if phi_modificato_template not in mappa_vettori_template:
                            continue
                            
                        dr_sub, dc_sub, k_sub, w_sub = mappa_vettori_template[phi_modificato_template]

                        # Eq. 18 e 19: Rotazione e scala geometrica dei vettori trasformati
                        dr_trasformati = s * (dr_sub * cos_t - dc_sub * sin_t)
                        dc_trasformati = s * (dr_sub * sin_t + dc_sub * cos_t)

                        centri_r = r_scena_sub[:, None] + dr_trasformati[None, :]
                        centri_c = c_scena_sub[:, None] + dc_trasformati[None, :]
                        
                        # STRATEGIA DI INCREMENTO DI BALLARD (Eq. 24 + Eq. 26 + Sez. 5.2)
                        # Modulo del gradiente g(x) * peso informatività Curvatura K * peso sottoparte w_i
                        pesi_voto = mag_scena_sub[:, None] * (k_sub[None, :] * w_sub[None, :])

                        centri_r_int = np.round(centri_r).astype(np.int32)
                        centri_c_int = np.round(centri_c).astype(np.int32)
                        
                        validi = (centri_r_int >= 0) & (centri_r_int < altezza) & (centri_c_int >= 0) & (centri_c_int < larghezza)
                        np.add.at(acc_temporaneo_voto, (centri_r_int[validi], centri_c_int[validi]), pesi_voto[validi])
                    
                    acc_2d_corrente = accumulatore_4d[:, :, idx_s, idx_theta]
                    if np.max(acc_temporaneo_voto) > np.max(acc_2d_corrente):
                        accumulatore_4d[:, :, idx_s, idx_theta] = acc_temporaneo_voto
                        meta_riflessione[(idx_s, idx_theta)] = "Specchiato" if riflettere == 1 else "Normale"

    miglior_punteggio_scena = 0.0
    miglior_scala = None
    miglior_rotazione = None
    tipo_riflessione_finale = "Normale"

    for idx_s in range(num_scale):
        for idx_theta in range(num_rotazioni):
            acc_piano = accumulatore_4d[:, :, idx_s, idx_theta]
            
            # Compensazione degli errori tramite filtro di smoothing (Sezione 2.2, Eq. 5)
            acc_piano_smooth = cv2.GaussianBlur(acc_piano, (5, 5), 0)
            _, max_voti_temp, _, _ = cv2.minMaxLoc(acc_piano_smooth)
            
            # Normalizzazione formale basata sull'energia cumulabile totale della R-table
            punteggio_corrente = (max_voti_temp / (vettori_totali_conteggio * 255)) if vettori_totali_conteggio > 0 else 0
            
            if punteggio_corrente > miglior_punteggio_scena:
                miglior_punteggio_scena = punteggio_corrente
                miglior_scala = scale_test[idx_s]
                miglior_rotazione = rotazioni_test[idx_theta]
                tipo_riflessione_finale = meta_riflessione.get((idx_s, idx_theta), "Normale")

    t_fine_scena = time.time() - t_scena
    
    if miglior_punteggio_scena >= soglia_confidenza:
        immagini_con_oggetto.append(nome_scena)
        print(f"[TROVATO] '{nome_scena}' - Confidenza Formale Ballard: {miglior_punteggio_scena:.2%}")
        print(f"          > Scala: {miglior_scala} | Rotazione: {miglior_rotazione}° | Assetto: {tipo_riflessione_finale} | Tempo: {t_fine_scena:.2f}s\n")
    else:
        print(f"[NON TROVATO] '{nome_scena}' - Escluso (Confidenza: {miglior_punteggio_scena:.2%}) | Tempo: {t_fine_scena:.2f}s\n")

print("\n==================================================")
print(f" REPORT FINALE BALLARD VALIDATO - Tempo Totale: {time.time() - t_inizio_totale:.2f}s")
print(f" Totale immagini analizzate: {totale_immagini}")
print(f" Immagini positive identificate: {len(immagini_con_oggetto)}")
print("==================================================")


--- AVVIO ACCUMULATORE FORMALE BALLARD ---
[NON TROVATO] 'Bone-3.gif' - Escluso (Confidenza: 13.77%) | Tempo: 1.50s

[NON TROVATO] 'apple-1.gif' - Escluso (Confidenza: 2.99%) | Tempo: 0.77s

[NON TROVATO] 'apple-2.gif' - Escluso (Confidenza: 3.17%) | Tempo: 0.75s

[NON TROVATO] 'apple-3.gif' - Escluso (Confidenza: 3.23%) | Tempo: 0.90s

[NON TROVATO] 'apple-4.gif' - Escluso (Confidenza: 2.98%) | Tempo: 0.76s

[NON TROVATO] 'apple-5.gif' - Escluso (Confidenza: 3.19%) | Tempo: 0.99s

[NON TROVATO] 'apple-6.gif' - Escluso (Confidenza: 3.81%) | Tempo: 0.85s



KeyboardInterrupt: 

### SEZIONE CRITICITÀ E LIMITAZIONI

*   **Sensibilità alle variazioni di scala e rotazione:** La R-Table classica è rigidamente vincolata alla scala e all'orientamento dei template di addestramento. L'estensione dell'algoritmo a varianti scalate o ruotate comporterebbe un incremento dimensionale dello spazio di accumulazione (introducendo assi per i fattori di scala $S$ e di rotazione $\theta$), inficiando l'efficienza computazionale del broadcasting.
*   **Arbitrarietà delle soglie operative:** La scelta di applicare soglie di magnitudo asimmetriche (superiore a 50 per l'addestramento e superiore a 100 per l'analisi della scena) risponde a un'esigenza empirica di filtraggio del rumore ambientale, ma introduce una forte dipendenza dalle condizioni di illuminazione e di contrasto delle immagini di test.
*   **Confronto con architetture di Deep Learning:** Rispetto a modelli di Object Detection moderni basati su Convolutional Neural Networks o Vision Transformers (ad esempio RetinaNet), l'approccio geometrico classico soffre in presenza di occlusioni parziali o texture complesse. Architetture basate su Focal Loss **riducono significativamente** i falsi positivi sbilanciando i gradienti di errore verso gli esempi negativi difficili (*hard negatives*) durante il training, una proprietà nativamente preclusa alla natura puramente additiva della Generalized Hough Transform.